<a href="https://colab.research.google.com/github/marianamachaddo/Prog26/blob/main/atividade_pratica_aula3_limpeza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Prática — Aula 3: Limpeza, Preparação e Qualidade dos Dados com Pandas

Esta atividade foi construída com base nos slides da Aula 3, que destacam a **limpeza como a fundação invisível** de qualquer dashboard confiável. O objetivo aqui não é apenas "fazer funcionar", mas tomar decisões conscientes sobre tipos, valores ausentes, duplicidades, variáveis derivadas e exportação da base limpa. fileciteturn2file0

## Regras desta atividade
- Você deve **construir os códigos**.
- O notebook orienta os passos, mas não entrega as soluções prontas.
- Sempre que fizer uma decisão de limpeza, **documente o porquê** em comentário ou célula markdown.
- Ao final, exporte uma base limpa para uso nas próximas aulas.

## Dataset desta atividade
Arquivo bruto: `vendas_brasil_aula3_bruto.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias para trabalhar com:
- leitura de dados
- tratamento de nulos
- identificação de infinitos
- exportação do resultado final

**Sugestão:** Pandas e NumPy já resolvem toda a atividade.


In [3]:
import pandas as pd
import numpy as np

## 2. Leitura da base bruta

Leia o arquivo `vendas_brasil_aula3_bruto.csv` em um DataFrame chamado `df`.

Depois:
1. Exiba as primeiras linhas
2. Exiba as últimas linhas
3. Observe visualmente se já existem sinais de sujeira


In [4]:
# Ler o CSV
df = pd.read_csv('vendas_brasil_aula3_bruto.csv')

# Primeiras linhas
df.head()

# Últimas linhas
df.tail()

,pedido_id,data,uf,canal,categoria,produto,cliente_id,quantidade,receita,lucro,observacao
223,1180,2024-03-11,SC,Loja Física,Telefonia,Smartphone X,C158,1,2563.44,415.42,NaN
224,1015,2024-02-26,RS,Online,Informática,Notebook Pro,C102,3,15011.49,NaN,cliente premium
225,1115,2024-07-29,MG,Loja Física,Informática,Notebook Pro,C144,3,12566.04,2879.76,NaN
226,1172,2024-11-11,ES,Loja Física,Informática,Notebook Pro,C128,3,13154.04,3408.22,entrega rápida
227,1209,2024-05-27,MG,Marketplace,Áudio,Caixa de Som,C121,2,810.2,282.52,NaN


## 3. Check-up inicial do dataset

Com base no checklist de um dataset "clean" apresentado na aula, faça um diagnóstico inicial da base. fileciteturn2file0

### Sua missão
Use comandos que permitam responder:
- Qual é o tamanho da base?
- Quais são os tipos atuais das colunas?
- Existem valores nulos?
- Há colunas com tipo inadequado?
- Há algo que pareça impedir análises confiáveis?


In [5]:
# Tamanho da base
df.shape

# Informações gerais
df.info()

# Tipos das colunas
df.dtypes

# Valores nulos
df.isna().sum()

# Estatísticas
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228 entries, 0 to 227
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   pedido_id   228 non-null    int64  
 1   data        228 non-null    object 
 2   uf          223 non-null    object 
 3   canal       223 non-null    object 
 4   categoria   223 non-null    object 
 5   produto     228 non-null    object 
 6   cliente_id  228 non-null    object 
 7   quantidade  228 non-null    int64  
 8   receita     226 non-null    object 
 9   lucro       222 non-null    float64
 10  observacao  178 non-null    object 
dtypes: float64(1), int64(2), object(8)
memory usage: 19.7+ KB


,pedido_id,quantidade,lucro
count,228.000000,228.000000,222.000000
mean,1110.324561,2.986842,1333.256712
std,63.563697,1.189085,1352.837197
min,1000.000000,1.000000,106.780000
25%,1055.750000,2.000000,385.005000
50%,1111.500000,3.000000,750.700000
75%,1165.250000,4.000000,1925.122500
max,1219.000000,6.000000,6335.030000


## 4. Batalha 1 — A tirania dos tipos de dados

Nos slides, vimos que datas não podem ser tratadas como texto e que números em formato string quebram análises. fileciteturn2file0

### Tarefa
Converta corretamente, quando fizer sentido:
- `data`
- `receita`
- `lucro`
- `quantidade`

### Orientação
- Investigue valores estranhos antes de converter
- Pense no uso de `errors='coerce'`
- Registre em markdown ou comentário quais problemas encontrou


In [6]:
df['data'].unique()
df['receita'].unique()
df['lucro'].unique()
df['quantidade'].unique()


array([1, 2, 5, 4, 3, 6])

In [7]:
# Converter data
df['data'] = pd.to_datetime(df['data'], errors='coerce')

# Converter números
df['receita'] = pd.to_numeric(df['receita'], errors='coerce')
df['lucro'] = pd.to_numeric(df['lucro'], errors='coerce')
df['quantidade'] = pd.to_numeric(df['quantidade'], errors='coerce')

### Reflexão curta
Explique:
1. Por que deixar `data` como texto pode quebrar análises temporais?
2. Por que deixar `receita` como string pode distorcer cálculos?


1. Porque não permite ordenar corretamente, extrair mês/ano ou fazer análises por período.
2. Porque não é possível somar, calcular média ou indicadores financeiros corretamente.

## 5. Batalha 2 — O enigma dos valores ausentes

A aula reforça que valores ausentes não devem ser tratados automaticamente; a decisão precisa ser justificada. fileciteturn2file0

### Tarefa
1. Mapeie os valores ausentes por coluna
2. Identifique quais colunas críticas têm nulos
3. Defina uma estratégia para cada caso:
   - remover linhas?
   - preencher?
   - manter?
4. Justifique cada escolha

### Dica
Evite decisões automáticas sem análise de contexto.


In [8]:
# Mapear nulos
df.isna().sum()


,0
pedido_id,0
data,224
uf,5
canal,5
categoria,5
produto,0
cliente_id,0
quantidade,0
receita,15
lucro,6


In [9]:
# Exemplo de decisões
df = df.dropna(subset=['data'])  # remover se data for nula
df['quantidade'] = df['quantidade'].fillna(0)
df['lucro'] = df['lucro'].fillna(0)

### Registro reflexivo
Escreva aqui sua justificativa para o tratamento dos nulos:
- Em quais colunas você removeu linhas?
- Em quais colunas você preencheu valores?
- Em quais situações decidiu manter o nulo?



* Removi linhas sem data porque não permitem análise temporal.
* Preenchi quantidade e lucro com 0 porque podem representar vendas sem lucro.
* Mantive nulos em categorias para investigar depois.

## 6. Batalha 3 — O ataque dos clones (duplicidades)

A aula alerta que nem toda duplicidade é erro automático: pode haver compra legítima repetida ou dupla inserção do sistema. fileciteturn2file0

### Tarefa
1. Investigue se há linhas duplicadas
2. Analise se a duplicidade parece nociva para os KPIs
3. Escolha uma estratégia:
   - remover duplicidades totais?
   - remover apenas com base em certas colunas?
   - manter?
4. Justifique a decisão


In [10]:
# Ver duplicados
df.duplicated().sum()

# Ver quais são
df[df.duplicated()]

# Remover duplicados
df = df.drop_duplicates()

## 7. Padronização de categorias

Os slides mostram que categorias duplicadas ou mal escritas podem distorcer rankings e filtros. fileciteturn2file0

### Tarefa
Inspecione e padronize, se necessário:
- `uf`
- `canal`
- `categoria`

### Pense em:
- maiúsculas e minúsculas
- espaços extras
- acentuação / variações
- categorias equivalentes escritas de formas diferentes


In [12]:
# Padronizar texto
df['uf'] = df['uf'].str.upper().str.strip()
df['canal'] = df['canal'].str.upper().str.strip()
df['categoria'] = df['categoria'].str.upper().str.strip()

## 8. Subindo de nível — Criação de variáveis derivadas

Depois da limpeza, é hora de enriquecer a base com variáveis úteis para análise. fileciteturn2file0

### Tarefa
Crie, se possível:
- `ano`
- `mes`
- `ano_mes`
- `margem_lucro`

### Atenção
A criação de `margem_lucro` exige cuidado com divisões por zero e possíveis valores infinitos.


In [13]:
# Ano e mês
df['ano'] = df['data'].dt.year
df['mes'] = df['data'].dt.month

# Ano_mes
df['ano_mes'] = df['data'].dt.to_period('M')

# Margem de lucro
df['margem_lucro'] = df['lucro'] / df['receita']

# Tratar infinitos
df['margem_lucro'] = df['margem_lucro'].replace([np.inf, -np.inf], np.nan)

### Reflexão técnica
Explique:
1. O que pode acontecer ao calcular margem de lucro quando a receita é zero?
2. Como você decidiu tratar esse caso?


1. Divisão por zero gera infinito.
2. Substituir infinito por NaN ou 0.

## 9. Seleção final — Menos é mais

A aula propõe que nem toda coluna precisa seguir para a base analítica final. fileciteturn2file0

### Tarefa
Crie um DataFrame final, por exemplo `df_clean` ou `df_dash`, contendo apenas as colunas estritamente necessárias para análises de negócio.

**Sugestão de raciocínio:**
- Quais colunas ajudam a responder perguntas de negócio?
- Quais colunas são operacionais, auxiliares ou dispensáveis?
- O que precisa existir para as próximas aulas de visualização e dashboard?


In [14]:
df_clean = df[[
    'data',
    'ano',
    'mes',
    'ano_mes',
    'uf',
    'canal',
    'categoria',
    'quantidade',
    'receita',
    'lucro',
    'margem_lucro'
]]


## 10. Validação final da base limpa

Antes de exportar, faça uma checagem final.

### Verifique:
- os tipos estão corretos?
- ainda há nulos problemáticos?
- ainda há duplicidades nocivas?
- existe algum infinito em `margem_lucro`?
- a base está pronta para ser usada em análises e dashboards?


In [15]:
df_clean.info()
df_clean.isna().sum()
df_clean.duplicated().sum()
np.isinf(df_clean['margem_lucro']).sum()


<class 'pandas.core.frame.DataFrame'>
Index: 4 entries, 0 to 154
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   data          4 non-null      datetime64[ns]
 1   ano           4 non-null      int32         
 2   mes           4 non-null      int32         
 3   ano_mes       4 non-null      period[M]     
 4   uf            4 non-null      object        
 5   canal         4 non-null      object        
 6   categoria     4 non-null      object        
 7   quantidade    4 non-null      int64         
 8   receita       3 non-null      float64       
 9   lucro         4 non-null      float64       
 10  margem_lucro  3 non-null      float64       
dtypes: datetime64[ns](1), float64(3), int32(2), int64(1), object(3), period[M](1)
memory usage: 352.0+ bytes


np.int64(0)

## 11. Exportação da base "Clean"

Agora gere o arquivo final da base tratada.

### Tarefa
Exporte sua base limpa com o nome:

`vendas_brasil_aula3_clean.csv`

### Observação
Esse arquivo será o selo de qualidade do seu trabalho desta aula.


In [16]:
df_clean.to_csv('vendas_brasil_aula3_clean.csv', index=False)

## 12. Conclusão e registro reflexivo

Escreva um pequeno texto respondendo:

1. Quais foram os principais problemas de qualidade encontrados?
2. Qual decisão de limpeza foi mais difícil?
3. Que impacto essas falhas poderiam causar em um dashboard?
4. Por que a etapa de limpeza é considerada a "fundação invisível" do projeto?


Os principais problemas encontrados foram tipos de dados incorretos, valores nulos, duplicidades e categorias escritas de formas diferentes.
A decisão mais difícil foi tratar valores ausentes, pois foi necessário decidir entre remover linhas ou preencher valores.
Essas falhas poderiam causar erros em cálculos de receita, lucro e análises temporais, além de dashboards incorretos.
A limpeza é considerada a fundação invisível porque, sem dados limpos, qualquer análise ou dashboard pode gerar conclusões erradas.

## 13. Desafio extra (opcional)

Use sua base limpa para responder rapidamente:

- Qual UF concentra maior receita?
- Qual canal gera maior lucro?
- Existe algum mês com desempenho claramente acima dos demais?

Você pode responder com tabelas simples ou pequenos agrupamentos.


In [17]:
# UF com maior receita
df_clean.groupby('uf')['receita'].sum().sort_values(ascending=False)

# Canal com maior lucro
df_clean.groupby('canal')['lucro'].sum().sort_values(ascending=False)

# Receita por mês
df_clean.groupby('ano_mes')['receita'].sum().sort_values(ascending=False)

,receita
ano_mes,
2024-10,11466.36
2024-04,4535.11
2024-09,3998.83
